# Notebook 3: Silver Layer - Cleaning and Standardization

**Objective**: Transform raw data from the Bronze layer into clean and standardized data.

## What is the Silver Layer?

The **Silver Layer** is the second layer of the lakehouse:
- **Cleaning**: Removal of invalid data
- **Standardization**: Consistent column names (snake_case)
- **Typing**: Conversion to appropriate data types
- **Deduplication**: Removal of duplicates

## Data Flow

```
Bronze (raw) ──► Filtering ──► Standardization ──► Silver (clean)
```

---
## INITIALIZATION

In [1]:
# Configure path to project
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark

# Create Spark session
spark = get_spark("silver-transformations")

# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

print("=" * 60)
print("SILVER LAYER - CLEANING AND STANDARDIZATION")
print("=" * 60)

SILVER LAYER - CLEANING AND STANDARDIZATION


## 1. Reading the Bronze Table

The Silver layer reads from Bronze, never from the original source.

In [2]:
# Read Bronze table
sales_bronze = spark.table("nessie.bronze.sales")

bronze_count = sales_bronze.count()
print(f"Records in Bronze: {bronze_count:,}")

# Data preview
print("\n=== Bronze Preview ===")
sales_bronze.select("Order ID", "Order Date", "Customer Name", "Sales", "Quantity").show(5, truncate=False)

Records in Bronze: 10,365

=== Bronze Preview ===
+--------------+----------+-------------+------+--------+
|Order ID      |Order Date|Customer Name|Sales |Quantity|
+--------------+----------+-------------+------+--------+
|CA-2020-124499|2020-10-08|Fred McMath  |389.97|3       |
|CA-2020-162761|2020-10-08|Sonia Cooley |11.214|2       |
|CA-2020-124107|2020-10-08|Brian Moss   |29.16 |3       |
|CA-2020-162761|2020-10-08|Sonia Cooley |1.872 |2       |
|CA-2020-141040|2020-10-08|Tim Brockman |631.96|4       |
+--------------+----------+-------------+------+--------+
only showing top 5 rows



## 2. Standardization of Column Names

Conversion from **PascalCase** (Bronze) to **snake_case** (Silver):

| Bronze | Silver |
|--------|--------|
| `Order ID` | `order_id` |
| `Order Date` | `order_date` |
| `Customer Name` | `customer_name` |
| `Sales` | `sales` |

In [3]:
from pyspark.sql.functions import col

# Rename columns and standardize
sales_silver = (
    sales_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        # Preserved metadata
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
)

print("=== Standardized data ===")
sales_silver.select("order_id", "order_date", "customer_name", "sales", "quantity").show(5, truncate=False)

=== Standardized data ===
+--------------+----------+-------------+------+--------+
|order_id      |order_date|customer_name|sales |quantity|
+--------------+----------+-------------+------+--------+
|CA-2020-124499|2020-10-08|Fred McMath  |389.97|3       |
|CA-2020-162761|2020-10-08|Sonia Cooley |11.214|2       |
|CA-2020-124107|2020-10-08|Brian Moss   |29.16 |3       |
|CA-2020-162761|2020-10-08|Sonia Cooley |1.872 |2       |
|CA-2020-141040|2020-10-08|Tim Brockman |631.96|4       |
+--------------+----------+-------------+------+--------+
only showing top 5 rows



## 3. Data Quality - Filtering

Removal of invalid records:
- NULL values on critical fields (order_id, product_id, sales)
- Duplicates (same order_id + product_id)

In [4]:
# Filter invalid data
sales_silver_clean = (
    sales_silver
    # Remove NULL values on critical fields
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    # Deduplication on natural key
    .dropDuplicates(["order_id", "product_id"])
)

silver_count = sales_silver_clean.count()
filtered_count = bronze_count - silver_count

print(f"Bronze: {bronze_count:,} records")
print(f"Silver: {silver_count:,} records")
print(f"Filtered: {filtered_count:,} records ({filtered_count/bronze_count*100:.2f}%)")

Bronze: 10,365 records
Silver: 9,917 records
Filtered: 448 records (4.32%)


## 4. Data Quality Verification

Controls to ensure that Silver data is clean.

In [5]:
from pyspark.sql.functions import sum, when, count as spark_count

# Check for NULL values
null_checks = sales_silver_clean.select(
    spark_count(when(col("order_id").isNull(), 1)).alias("null_order_id"),
    spark_count(when(col("product_id").isNull(), 1)).alias("null_product_id"),
    spark_count(when(col("sales").isNull(), 1)).alias("null_sales"),
)

print("=== Quality checks ===")
print("NULL values:")
null_checks.show()

# Check for negative values
neg_sales = sales_silver_clean.filter(col("sales") < 0).count()
neg_quantity = sales_silver_clean.filter(col("quantity") <= 0).count()
invalid_discount = sales_silver_clean.filter((col("discount") < 0) | (col("discount") > 1)).count()

print(f"Negative sales: {neg_sales}")
print(f"Invalid quantities (<= 0): {neg_quantity}")
print(f"Invalid discounts: {invalid_discount}")

=== Quality checks ===
NULL values:
+-------------+---------------+----------+
|null_order_id|null_product_id|null_sales|
+-------------+---------------+----------+
|            0|              0|         0|
+-------------+---------------+----------+

Negative sales: 0
Invalid quantities (<= 0): 36
Invalid discounts: 33


## 5. Dropping the old Silver table (if it exists)

In [6]:
# Cleanup: drop table if it exists
spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")

print("Existing tables dropped")

Existing tables dropped


## 6. Creating the Silver table

In [7]:
# Create Silver table
sales_silver_clean.writeTo("nessie.silver.sales").using("iceberg").create()

print("Silver table created: nessie.silver.sales")

Silver table created: nessie.silver.sales


## 7. Verification and example queries

In [8]:
# Verify the count
final_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.sales").first()[0]
print(f"Records in Silver: {final_count:,}")

Records in Silver: 9,917


In [9]:
# Example analysis: Sales by category
print("=== Sales by category ===")
spark.sql("""
    SELECT 
        category,
        ROUND(SUM(sales), 2) as total_sales,
        SUM(quantity) as total_quantity,
        COUNT(DISTINCT order_id) as order_count
    FROM nessie.silver.sales
    GROUP BY category
    ORDER BY total_sales DESC
""").show(truncate=False)

=== Sales by category ===
+---------------+-----------+--------------+-----------+
|category       |total_sales|total_quantity|order_count|
+---------------+-----------+--------------+-----------+
|Technology     |828694.06  |6862          |1536       |
|Furniture      |734770.98  |7940          |1754       |
|Office Supplies|716017.89  |22624         |3726       |
+---------------+-----------+--------------+-----------+



In [10]:
# Distribution by batch
print("=== Distribution by batch ===")
spark.sql("""
    SELECT 
        batch_id,
        COUNT(*) as count,
        ROUND(SUM(sales), 2) as total_sales
    FROM nessie.silver.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show(truncate=False)

=== Distribution by batch ===
+--------+-----+-----------+
|batch_id|count|total_sales|
+--------+-----+-----------+
|batch_1 |3260 |760753.3   |
|batch_2 |3327 |783750.44  |
|batch_3 |3330 |734979.18  |
+--------+-----+-----------+



In [11]:
# Top 5 products
print("=== Top 5 products ===")
spark.sql("""
    SELECT 
        category,
        product_name,
        ROUND(SUM(sales), 2) as total_sales,
        SUM(quantity) as total_quantity
    FROM nessie.silver.sales
    GROUP BY category, product_name
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

=== Top 5 products ===
+---------------+---------------------------------------------------------------------------+-----------+--------------+
|category       |product_name                                                               |total_sales|total_quantity|
+---------------+---------------------------------------------------------------------------+-----------+--------------+
|Technology     |Canon imageCLASS 2200 Advanced Copier                                      |61599.82   |20            |
|Office Supplies|Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind|27453.38   |31            |
|Technology     |Cisco TelePresence System EX90 Videoconferencing Unit                      |22638.48   |6             |
|Furniture      |HON 5400 Series Task Chairs for Big and Tall                               |21870.58   |39            |
|Office Supplies|GBC DocuBind TL300 Electric Binding System                                 |19823.48   |37            |
+--------

## SILVER LAYER SUMMARY

In [14]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         SILVER LAYER - TRANSFORMATION COMPLETED                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Table               nessie.silver.sales                                 ║
║  Format              Iceberg                                             ║
║                                                                          ║
║  Transformations:                                                        ║
║    - Columns renamed (snake_case)                                        ║
║    - Types converted (double, int)                                       ║
║    - NULL filtered on critical fields                                    ║
║    - Duplicates removed                                                  ║
║                                                                          ║
║  Volumes:                                                                ║
║    Bronze: {:>6,} records                                                ║
║    Silver: {:>6,} records                                                ║
║    Filtered: {:>10,} ({:>5.1f}%)                                         ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""".format(bronze_count, final_count, filtered_count, filtered_count/bronze_count*100))

print("→ Next step: Run '04_gold_layer.ipynb'")


╔══════════════════════════════════════════════════════════════════════════╗
║         SILVER LAYER - TRANSFORMATION COMPLETED                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Table               nessie.silver.sales                                 ║
║  Format              Iceberg                                             ║
║                                                                          ║
║  Transformations:                                                        ║
║    - Columns renamed (snake_case)                                        ║
║    - Types converted (double, int)                                       ║
║    - NULL filtered on critical fields                                    ║
║    - Duplicates removed                                                  ║
║                                                                          